# Analyse d'hyper-paramètres

## Chargement de la dataset

In [ ]:
from datasets.load import load_msrcv1

name_dataset = "msrcv1"
dataset = load_msrcv1("datasets/")
k = 7
dataset.keys()

In [ ]:
from bagging import bagging_prime
from metriques import clusteringMeasure

## Fine Tuning des hyper-paramètres 

l'objectif ici est de trouver les valeurs par défaut du modèle

Définitions des fonctions utiles

In [ ]:
from numpy import mean


def execution(X, y, combinaison, typeweak, nexec):
    p = combinaison[0]
    nw = combinaison[1]

    perfs = {"ACC": [], "NMI": [], "PUR": []}
    for _ in range(nexec):
        r = bagging_prime(X, k, typeweak=typeweak, nbreweak=nw, p=p)
        perf = clusteringMeasure(y, r)
        for key in perfs.keys():
            perfs[key].append(perf[key])

    for key in perfs.keys():
        perfs[key] = mean(perfs[key])

    perfs["p"] = p
    perfs["nw"] = nw

    return perfs

In [ ]:
from itertools import product

def experiment_analyse_base(dataset, Ps, Nws, typeweak, nbexec=10):
    combinaisons = list(product(Ps, Nws))

    results = []
    for combinaison in combinaisons:
        res = execution(dataset["X"], dataset["Y"], combinaison, typeweak, nbexec)
        res["typeweak"] = typeweak
        results.append(res)
        print(res)

    return results

In [ ]:
from pandas import DataFrame
def save_experiment(results, path_name):
    df = DataFrame(results).sort_values(by=["ACC", "NMI", "PUR"], ascending=False)
    df.to_csv(path_name)
    return df

Définition de la grille de recherche

In [ ]:
from numpy import arange

# variables
Ps = arange(0.5, 1, 0.05)
Nws = arange(10, 100, 5)

In [ ]:
typeweak = "mvgl"
results_mvgl = experiment_analyse_base(dataset, Ps, Nws, typeweak)

In [ ]:
save_experiment(results_mvgl, f"results/fine_tuning.{typeweak}.{name_dataset}.csv")

## Performances suivant la progréssion des hyper-paramètres

L'objectif ici est de voir comment les performances du modèle évoluent, par rapport à des augmentations de valeurs d'hyper-paramètres

### Taille des échantillons (pourcentage)

In [ ]:
Ps = arange(0.5,1,0.05)
Nws = [100]

In [ ]:
typeweak = "mcles"
results_mcles = experiment_analyse_base(dataset, Ps, Nws, typeweak)

In [ ]:
save_experiment(results_mcles, f"results/annalyse_p.{typeweak}.{name_dataset}.csv")

In [ ]:
typeweak = "mvgl"
results_mvgl = experiment_analyse_base(dataset, Ps, Nws, typeweak)

In [ ]:
save_experiment(results_mvgl, f"results/annalyse_p.{typeweak}.{name_dataset}.csv")

### Nombre de weakleaners

In [ ]:
Nws = arange(10, 101, 5)

In [ ]:
typeweak = "mcles"
Ps = [0.6]
results_mcles = experiment_analyse_base(dataset, Ps, Nws, typeweak, nbexec=5)

In [ ]:
save_experiment(results_mcles, f"results/annalyse_nw.{typeweak}.{name_dataset}.csv")

In [ ]:
typeweak = "mvgl"
Ps = [0.55]
Nws = arange(40, 101, 5)
results_mvgl = experiment_analyse_base(dataset, Ps, Nws, typeweak)

In [ ]:
save_experiment(results_mvgl, f"results/annalyse_nw.{typeweak}.{name_dataset}.csv")